# Neuron-Level Causal Validation of Modality Competition — RML (v3)

Implements `experiment_protocol[From Prof KC Lan].docx` end to end. Section numbering follows that protocol's **Day 0 / Week 1–4** schedule, not the older "Phase A–E" labels.

**Restart rationale:** v1 and v2 evaluated the correct, SHA-verified RML checkpoints against a fabricated `meta.pkl` that does not exist in the source data archive. All prior numbers are retracted. See `docs/adr/0004-v3-restart-data-provenance-and-fidelity-gate.md`, `docs/adr/0005-day0-dominant-modality-recompute-and-fast-path.md`, and `journals/2026-08-09.md`.

**What is different in v3:**

| | |
|---|---|
| **Canonical data** | `BIG_DATA_RAW_PROCESSED_FACE` + `meta_one_hot_label_six_categories.pkl` filtered to the RML split files — the same source `main.py` used to train and evaluate every dataset in the thesis. The retired `RML_RAW_PROCESSED_Face` folder is never touched. |
| **Halting gates, not warnings** | Every check below `assert`s. Nothing prints a warning and continues. There is **no label-permutation solver** — the label order is the fixed constant from `getEmotionDict()`. |
| **Day 0 integrity gate** | Replicates `sample_imgs_by_interval` and confirms every constructed frame path exists, before any model runs. |
| **Day 0 fidelity gate** | Both checkpoints must reproduce 76.39% / 79.86% test accuracy (from the original training logs, now in `docs/provenance/`) within 1pp, or nothing downstream runs. |
| **Day 0 dominant modality** | Recomputed from the DeepSHAP pickles rather than inherited, and cross-checked in Week 1 against the activations actually extracted. |
| **Falsification pair** | A full-64 knockout and a random-5 control run *before* any real sweep, plus a threshold-free mechanical proof that the ablation hook writes through. |
| **Exact fast path** | Weeks 2–3 replay the model's linear head over cached activations instead of re-reading the dataset ~100 times, after proving it reproduces real forward passes exactly. |

**Run order:** top to bottom on a Colab GPU runtime. Do not skip a gate. If a cell halts, read its message — every `HALT` says what to check and what not to assume.

# Day 0 — Environment & Path Setup

In [ ]:
# -- Torch/Torchaudio/Torchvision ABI self-check & auto-repair (Colab) --
# Colab's preinstalled torch/torchaudio/torchvision trio can drift out of sync
# across runtime image updates (mismatched AOTInductor ABI -> `undefined symbol:
# aoti_torch_abi_version` on `import torchaudio`). Rather than pin exact versions
# here (old pins may lack wheels for Colab's current Python), this cell verifies
# the trio actually imports together and, if not, reinstalls a mutually
# compatible set from PyPI and force-restarts the runtime once.
#
# Re-run this cell (and then all cells below) after any Colab runtime restart.
# See journals/2026-08-09.md for the incident this guards against.
import os
import subprocess
import sys

def _torch_stack_ok():
    try:
        import torch, torchaudio, torchvision  # noqa: F401
        return True
    except Exception as e:
        print(f"Torch stack import failed ({e!r}); reinstalling a matched trio.")
        return False

if not _torch_stack_ok():
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torch", "torchaudio", "torchvision"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio", "torchvision"],
        check=True,
    )
    print("Reinstalled matched torch/torchaudio/torchvision.", flush=True)
    print("RESTARTING RUNTIME NOW -- Colab reconnects on its own. When it does, "
          "re-run this notebook from cell 1.", flush=True)
    # Without an actual process kill the stale, already-imported torch stays
    # resident and every cell below fails with the same ABI error the reinstall
    # just fixed. The old version of this cell only printed "Restarting..." and
    # kept going, which is how that failure looked like a fix that didn't work.
    os.kill(os.getpid(), 9)

In [ ]:
# -- Global Environment & Path Setup --
import os
import sys
import random
import hashlib
import types
import builtins
import math
import numpy as np
import torch
import torch.nn as nn

builtins.torch = torch
builtins.nn = nn

# Defensive compatibility shims for transformers/torch API drift.
# These only patch missing attributes so transformers' import checks don't
# crash against whatever torch build the auto-repair cell above installed;
# they do not change any numerical behavior of the model itself.
if not hasattr(torch, 'get_default_device'):
    torch.get_default_device = lambda: torch.device("cpu")
if not hasattr(torch, 'set_default_device'):
    torch.set_default_device = lambda dev: None
if not hasattr(torch, 'is_compiling'):
    torch.is_compiling = lambda: False
if not hasattr(torch, 'compiler'):
    comp_mod = types.ModuleType('compiler')
    comp_mod.is_compiling = lambda: False
    torch.compiler = comp_mod

import transformers
import transformers.utils.import_utils as import_utils
import_utils.BACKENDS_MAPPING["torch"] = (lambda: True, "PyTorch library")
import_utils.is_torch_available = lambda: True
import_utils._torch_available = True
if hasattr(transformers, 'is_torch_available'):
    transformers.is_torch_available = lambda: True
try:
    from transformers.models.albert.modeling_albert import AlbertModel as RealAlbertModel
    transformers.AlbertModel = RealAlbertModel
    sys.modules['transformers'].AlbertModel = RealAlbertModel
except Exception as e:
    print(f"Notice on AlbertModel binding: {e}")

import subprocess

def _ensure_installed(module_name, pip_name):
    try:
        __import__(module_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

_ensure_installed("facenet_pytorch", "facenet-pytorch")
# AlbertTokenizer (slow, SentencePiece-backed) needs `sentencepiece`, which is
# not always preinstalled on Colab -- same drift risk as torch/torchaudio above.
_ensure_installed("sentencepiece", "sentencepiece")

def set_deterministic_seed(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_deterministic_seed(0)

# -- Path resolution & Google Drive mount --
project_path = None
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

for candidate in ['/content/drive/MyDrive/multimodal-causal-ablation',
                  '/content/multimodal-causal-ablation',
                  os.getcwd()]:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'checkpoints')):
        project_path = candidate
        break
if project_path is None:
    project_path = os.getcwd()

os.chdir(project_path)
if project_path not in sys.path:
    sys.path.insert(0, project_path)
model_src = os.path.join(project_path, 'Model/Dig-Data_Model-Main')
if model_src not in sys.path:
    sys.path.insert(0, model_src)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Working directory: {os.getcwd()}")
print(f"Runtime device: {device}")

# -- Fixed constants (do not fit these to data; see ADR 0004) --
# Label ordering is the training-time constant from Model/Dig-Data_Model-Main/src/datasets.py::getEmotionDict().
# There is no label-permutation solver in this notebook.
EMO_DICT = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']
RAW_EMO_KEYS = ['ang', 'dis', 'fea', 'hap', 'sad', 'sur']

# MODEL_ARGS confirmed against the exact training command lines, now copied into
# docs/provenance/RML_{origin,finetune}_training_log.txt (line 1 of each):
#   origin:   -lr=1e-5 -wd=1e-3 -ep=40 -mod=tav -bs=4 --img-interval=500 --early-stop=6  --loss=ce
#   finetune: -lr=5e-5         -ep=40 -mod=tav -bs=4 --img-interval=500 --early-stop=40 --loss=ce
#   both:     --model=mme2e --num-emotions=6 --trans-dim=64 --trans-nlayers=4
#             --trans-nheads=4 --text-lr-factor=10 --text-model-size=large --text-max-len=100
# The two runs differ only in optimizer/schedule settings, which are irrelevant at
# inference. Every ARCHITECTURE argument below is identical across both, so one
# MODEL_ARGS correctly instantiates both checkpoints. (An earlier version of this
# comment quoted only the finetune line -- a documentation gap, never a behavior bug.)
MODEL_ARGS = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
    'text_max_len': 100,
}

# Canonical data source (ADR 0004). Do not point this at the retired
# RML_RAW_PROCESSED_Face folder -- that folder's meta.pkl was fabricated
# and is not part of the source archive.
DATA_DIR = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')
MAIN_FOLDER = os.path.join(DATA_DIR, 'BIG_DATA_RAW_PROCESSED_FACE')
META_PATH = os.path.join(MAIN_FOLDER, 'meta_one_hot_label_six_categories.pkl')
SPLIT_DIR = os.path.join(DATA_DIR, 'data_split', 'all_single_label_six_category', 'with_valid')

CHECKPOINTS_DIR = os.path.join(project_path, 'checkpoints')
BASE_CKPT = os.path.join(CHECKPOINTS_DIR, 'base_model.pt')
FT_CKPT = os.path.join(CHECKPOINTS_DIR, 'finetuned_model.pt')

for d in [os.path.join(CHECKPOINTS_DIR, 'activations'),
          os.path.join(project_path, 'results'),
          os.path.join(project_path, 'figures')]:
    os.makedirs(d, exist_ok=True)

def compute_sha256(filepath):
    if not os.path.exists(filepath):
        return "FILE_NOT_FOUND"
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

print("Checkpoint SHA-256:")
print(f"  base_model.pt      {compute_sha256(BASE_CKPT)}")
print(f"  finetuned_model.pt {compute_sha256(FT_CKPT)}")
print("Expected (docs/adr/0004-...):")
print("  base_model.pt      565bc220d187f2286500481fdab4d3b3dc4f92a2006ffb8f02ca4c882bbd82db")
print("  finetuned_model.pt a4c1707f7bcc189d3103b42ea82c9d01f6ec2f992850e8cc1072f38f4dadd0de")
assert compute_sha256(BASE_CKPT) == "565bc220d187f2286500481fdab4d3b3dc4f92a2006ffb8f02ca4c882bbd82db", \
    "base_model.pt does not match the verified checkpoint. Stop and re-sync before continuing."
assert compute_sha256(FT_CKPT) == "a4c1707f7bcc189d3103b42ea82c9d01f6ec2f992850e8cc1072f38f4dadd0de", \
    "finetuned_model.pt does not match the verified checkpoint. Stop and re-sync before continuing."
print("Checkpoint identity verified.")

# Day 0 — Data Loading (Canonical Source) & Halting Fidelity Gate

Loads RML samples from `BIG_DATA_RAW_PROCESSED_FACE` + `meta_one_hot_label_six_categories.pkl`, filtered by the RML split files — mirroring `main.py::get_dataset_iemocap` exactly (same main folder, same meta file, same label dict). Evaluates both checkpoints on the 144-sample test split and **halts** if test accuracy doesn't reproduce the paper's published RML numbers.

In [ ]:
# -- Day 0: Canonical Data Load + Halting Fidelity Gate --
import glob
import json
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader
import transformers
from transformers import AlbertTokenizer

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

from src.datasets import IEMOCAP, collate_fn
from src.models.e2e import MME2E

assert os.path.exists(META_PATH), (
    f"Canonical meta file not found at {META_PATH}. "
    "Extract BIG_DATA_RAW_PROCESSED_FACE.tar.gz (or its RML-relevant subset) "
    "from the handover drive into Model/Dig-Data_Model-Main/data/ before continuing. "
    "See docs/adr/0004-v3-restart-data-provenance-and-fidelity-gate.md."
)

with open(META_PATH, 'rb') as f:
    meta = pickle.load(f)
print(f"Loaded canonical meta: {len(meta)} total utterances across all six datasets.")

def load_rml_split(phase):
    path = os.path.join(SPLIT_DIR, f'Final_{phase}_split_six_categories_RML.txt')
    ids = open(path).read().splitlines()
    missing = [uid for uid in ids if uid not in meta]
    if missing:
        raise KeyError(
            f"{len(missing)}/{len(ids)} RML {phase} split IDs are not keys in the canonical meta file "
            f"(e.g. {missing[:5]}). The merged corpus may have re-ID'd RML samples during construction; "
            "do not proceed with a partial-ID extraction (see ADR 0004's blocking check)."
        )
    return ids

train_ids = load_rml_split('train')
valid_ids = load_rml_split('valid')
test_ids = load_rml_split('test')
print(f"RML split sizes (all IDs confirmed present in canonical meta): "
      f"train={len(train_ids)} valid={len(valid_ids)} test={len(test_ids)}")
# Fidelity check on split composition itself. 518/58/144 is not just Table 4.2 --
# it is the literal stdout of the training run that produced these checkpoints
# (docs/provenance/RML_origin_training_log.txt, "# Train samples = 518" etc.),
# and those logs name these exact three split files.
assert (len(train_ids), len(valid_ids), len(test_ids)) == (518, 58, 144), (
    f"Split sizes {(len(train_ids), len(valid_ids), len(test_ids))} don't match the training "
    "log's 518/58/144. Stop and investigate before continuing."
)

# ---------------------------------------------------------------------------
# Day 0 data-integrity gate
#
# IEMOCAP.__getitem__ reads two things per sample: `audio.wav`, and a list of
# frame paths built by `sample_imgs_by_interval`, which is *file-count* based:
#
#     nums = len(glob(folder/*)) - 2
#     sampled = [f'image_{i}.jpg' for i in range(0, nums, int(500/1000*30))]
#
# The `- 2` is exact only because every utterance folder carries BOTH
# `audio.wav` and `audio_16000.wav` (verified across all 720 RML folders on
# 2026-08-09). Consequences this gate exists to catch:
#   * A stray `.DS_Store` -- this repo lives in `My Drive` on macOS, so Finder
#     can write one at any time -- shifts `nums` by one. No exception is raised;
#     the model just silently sees a different frame set than it did in training.
#   * A partially-synced Drive folder shortens the listing, same silent shift.
#   * Any constructed `image_{i}.jpg` that doesn't exist surfaces as an opaque
#     DataLoader *worker* crash from PIL, mid-epoch, with no useful message.
#   * `nums <= 0` makes `collate_fn` silently DROP the sample (it does
#     `if sampledImgs.shape[0] == 0: continue`), quietly shrinking the split.
#
# Checking the constructed paths directly catches all four in one loop. This is
# the ADR 0004 failure class one layer down: wrong data, no error.
# ---------------------------------------------------------------------------
IMG_INTERVAL = 500
IMG_FPS = 30
IMG_STEP = int(IMG_INTERVAL / 1000 * IMG_FPS)  # 15, matching datasets.py exactly

def _sampled_frame_paths(sample_folder):
    """Byte-for-byte replica of IEMOCAP.sample_imgs_by_interval."""
    files = glob.glob(f'{sample_folder}/*')
    nums = len(files) - 2
    sampled = [os.path.join(sample_folder, f'image_{i}.jpg')
               for i in range(0, nums, IMG_STEP)]
    return files, nums, sampled

def data_integrity_problems(uttr_ids):
    problems = []
    file_counts = {}
    total_frames = 0
    for uid in uttr_ids:
        folder = os.path.join(MAIN_FOLDER, uid)
        if not os.path.isdir(folder):
            problems.append((uid, 'sample folder missing'))
            continue
        files, nums, sampled = _sampled_frame_paths(folder)
        file_counts[uid] = len(files)
        names = [os.path.basename(f) for f in files]
        if 'audio.wav' not in names:
            problems.append((uid, 'audio.wav missing'))
        strays = [n for n in names
                  if n not in ('audio.wav', 'audio_16000.wav')
                  and not (n.startswith('image_') and n.endswith('.jpg'))]
        if strays:
            problems.append((uid, f'stray file(s) shift the frame count: {strays[:3]}'))
        if nums <= 0:
            problems.append((uid, f'no frames sampled (len(files)={len(files)}); '
                                  'collate_fn would silently DROP this sample'))
        absent = [os.path.basename(p) for p in sampled if not os.path.exists(p)]
        if absent:
            problems.append((uid, f'{len(absent)} sampled frame(s) absent, e.g. {absent[:3]}'))
        total_frames += len(sampled)
    return problems, file_counts, total_frames

_all_ids = train_ids + valid_ids + test_ids
_problems, _file_counts, _total_frames = data_integrity_problems(_all_ids)
assert not _problems, (
    f"HALT: {len(_problems)} data-integrity problem(s) across the 720 RML split IDs "
    f"(first 10: {_problems[:10]}). On Colab a Drive-mount sync glitch looks exactly like "
    "missing data -- wait for the mount to settle and re-run this cell before assuming "
    "data loss. A `.DS_Store` inside an utterance folder is the other common cause and "
    "must be deleted, not ignored: it changes which video frames the model sees. "
    "See docs/adr/0004-v3-restart-data-provenance-and-fidelity-gate.md."
)
print(f"Data-integrity gate PASSED: {len(_all_ids)} folders, every audio.wav and every one "
      f"of {_total_frames} sampled frame paths present; no stray files; no zero-frame samples.")

# Secondary diagnostic: compare per-folder file counts against the manifest
# recorded locally on 2026-08-09. Drift here is not fatal on its own (the path
# check above is what actually matters) but it means the folder contents changed
# since that snapshot, which is worth seeing before a multi-hour run.
_manifest_path = os.path.join(project_path, 'docs', 'rml_file_manifest.json')
if os.path.exists(_manifest_path):
    with open(_manifest_path) as f:
        _manifest = json.load(f)
    _drift = {uid: (_manifest[uid], _file_counts[uid])
              for uid in _file_counts
              if uid in _manifest and _manifest[uid] != _file_counts[uid]}
    if _drift:
        print(f"NOTE: per-folder file counts drifted from docs/rml_file_manifest.json for "
              f"{len(_drift)} folder(s) (uid: expected -> actual): "
              f"{dict(list(_drift.items())[:5])}. The path check above still passed, so this "
              "is informational -- but confirm nothing was added to those folders.")
    else:
        print("File-count manifest matches docs/rml_file_manifest.json exactly.")
else:
    print("NOTE: docs/rml_file_manifest.json not found; skipped the secondary count check.")

def build_loader(uttr_ids, batch_size=8):
    texts = [meta[uid]['text'] for uid in uttr_ids]
    labels = [meta[uid]['label'] for uid in uttr_ids]  # already one-hot in meta_one_hot_label_six_categories.pkl
    dataset = IEMOCAP(
        main_folder=MAIN_FOLDER,
        utterance_ids=uttr_ids,
        texts=texts,
        labels=labels,
        label_annotations=RAW_EMO_KEYS,
        img_interval=IMG_INTERVAL,
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2,
                       pin_memory=True, collate_fn=collate_fn)

# batch_size differs from the training run's -bs=4, which is safe here and not an
# oversight: WrappedTransformerEncoder passes a src_key_padding_mask built from
# the true per-sample lengths, and every BatchNorm runs on running statistics
# under model.eval(), so no sample's forward pass can see another sample's
# padding. Batch size therefore changes only float summation order.
test_loader = build_loader(test_ids)
assert len(test_loader.dataset) == 144, f"Test loader has {len(test_loader.dataset)} samples, expected 144."

tokenizer = AlbertTokenizer.from_pretrained('albert-large-v2')

def evaluate_accuracy(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            _, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
            text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                     max_length=MODEL_ARGS['text_max_len'],
                                     padding='max_length', truncation=True)
            text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
            imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
            specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
            logits = model(imgs, img_lens, specs, spec_lens, text_inputs)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_targets.extend(Y.argmax(-1).cpu().numpy())
    preds, targets = np.array(all_preds), np.array(all_targets)
    return float(np.mean(preds == targets)) * 100.0

# `position_ids` in AlbertModel's embeddings is `torch.arange(max_position_embeddings)`
# -- a deterministic, non-learned buffer recomputed at construction, identical in
# every AlbertModel ever instantiated. transformers versions disagree only on
# whether it is registered persistent (in state_dict) or not. This is environment
# drift in a buffer with no learnable content, so it is the one key allowed to
# differ -- in EITHER direction, since which side has it depends on whether the
# runtime's transformers is newer or older than the one that saved these
# checkpoints. Anything else still halts.
BENIGN_BUFFER_KEYS = {"T.albert.embeddings.position_ids"}

def load_model(ckpt_path):
    model = MME2E(args=MODEL_ARGS, device=device).to(device)
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
    expected = set(model.state_dict())
    missing = expected - set(state_dict)
    extra = set(state_dict) - expected
    assert missing <= BENIGN_BUFFER_KEYS, (
        f"HALT: checkpoint {ckpt_path} is missing weights "
        f"{sorted(missing - BENIGN_BUFFER_KEYS)}"
    )
    assert extra <= BENIGN_BUFFER_KEYS, (
        f"HALT: checkpoint {ckpt_path} has unexpected keys beyond known-benign buffers: "
        f"{sorted(extra - BENIGN_BUFFER_KEYS)}"
    )
    for k in extra:
        state_dict.pop(k)
    # strict=True whenever nothing is missing. If only the benign buffer is
    # missing, strict=False is the sole way to load at all -- and the assert
    # above has already proven that is the ONLY key involved.
    model.load_state_dict(state_dict, strict=not missing)
    if missing:
        print(f"  ({os.path.basename(ckpt_path)}: loaded with strict=False solely to tolerate "
              f"the non-learned buffer {sorted(missing)}; all learnable weights matched.)")
    return model

print(f"\nEnvironment: torch={torch.__version__}, transformers={transformers.__version__}, "
      f"numpy={np.__version__}")
print("\nLoading base and fine-tuned models (any key mismatch beyond the non-learned "
      "`position_ids` buffer halts)...")
base_model = load_model(BASE_CKPT)
ft_model = load_model(FT_CKPT)
print("Both checkpoints loaded. No missing weights; no unexpected keys beyond the benign buffer.")

print("\nEvaluating unablated test accuracy (fixed getEmotionDict() label order, no permutation search)...")
base_acc = evaluate_accuracy(base_model, test_loader)
ft_acc = evaluate_accuracy(ft_model, test_loader)

TARGET_BASE_ACC = 76.39  # docs/provenance/RML_origin_training_log.txt   -> "Test (29)  0.763889"
TARGET_FT_ACC = 79.86    # docs/provenance/RML_finetune_training_log.txt -> "Test (15)  0.798611"
TOLERANCE = 1.0          # 1.0pp ~= 1.44 test samples, i.e. +/-1 sample passes, +/-2 halts.

print(f"\n=== Day 0 Fidelity Gate ===")
print(f"Base model test accuracy:       {base_acc:.2f}%  (target {TARGET_BASE_ACC}% +/- {TOLERANCE}%)")
print(f"Fine-tuned model test accuracy: {ft_acc:.2f}%  (target {TARGET_FT_ACC}% +/- {TOLERANCE}%)")

assert abs(base_acc - TARGET_BASE_ACC) < TOLERANCE, (
    f"HALT: base model test accuracy {base_acc:.2f}% does not reproduce the training log's "
    f"{TARGET_BASE_ACC}%. Do not proceed to Week 1. Re-check data provenance (ADR 0004) "
    "before re-running this cell."
)
assert abs(ft_acc - TARGET_FT_ACC) < TOLERANCE, (
    f"HALT: fine-tuned model test accuracy {ft_acc:.2f}% does not reproduce the training log's "
    f"{TARGET_FT_ACC}%. Do not proceed to Week 1. Re-check data provenance (ADR 0004) "
    "before re-running this cell."
)
print("\nFidelity gate PASSED. Safe to proceed to Day 0 step 2, the falsification pair, and Week 1.")

# Day 0 (cont.) — Step 2: Identify the Dominant Modality from DeepSHAP

Protocol Day 0, step 2: *"Identify the dominant modality for RML from your existing DeepSHAP results (Section VI-C data)... if it's a bimodal combination, pick the single modality with the larger individual SHAP contribution."*

ADR 0004 carried ADR 0001's Audio selection forward without recomputing it. This cell recomputes it from the upstream DeepSHAP output for the two SHA-verified RML checkpoints (`checkpoints/RML_{origin,finetune}_SHAP_value.pkl`, copied from `Final_result/Origin_training/SHAP/SHAP_value/` on the handover drive).

The SHAP input space is the 1152-d concatenation `[text_cls 1024 | v_cls 64 | a_cls 64]` — i.e. **the same CLS representations this notebook ablates**, so Day 0's attribution space is literally the intervention space. That also satisfies protocol Day 0 steps 3–4 (name the hooked module, confirm its width): the module is `model.a_transformer`, its output is the 64-d audio CLS, and the falsification pair below proves the hook writes into it.

Two views of one measurement (`mean(|phi|)` *is* `sum(|phi|)` divided by the modality's dimension count):

- **Aggregate attribution mass** — `sum(|phi|)` over a modality's dimensions. Scales with dimension count, so text's 1024 dims are advantaged 16:1 over audio's 64.
- **Per-neuron attribution** — `mean(|phi|)`, dimension-invariant. This is the metric ADR 0001 Decision 1 specifies, and the one that matters when the intervention is per-neuron.

In [ ]:
# -- Day 0 step 2: Dominant modality, recomputed on canonical DeepSHAP output --
import io

SHAP_PKLS = {
    'base': os.path.join(CHECKPOINTS_DIR, 'RML_origin_SHAP_value.pkl'),
    'finetuned': os.path.join(CHECKPOINTS_DIR, 'RML_finetune_SHAP_value.pkl'),
}
for _name, _p in SHAP_PKLS.items():
    assert os.path.exists(_p), (
        f"Day 0 dominant-modality input missing: {_p}. These pickles are gitignored (checkpoints/*.pkl), so "
        "they reach Colab through Drive sync only. Copy them from the handover drive at "
        "Final_result/Origin_training/SHAP/SHAP_value/1225_RML_{origin,finetune}_*_SHAP_value.pkl "
        "and wait for the mount to finish syncing before re-running."
    )

class _CPUUnpickler(pickle.Unpickler):
    """The upstream pickles hold CUDA tensors saved on the original researchers'
    GPU box; without this they raise on any runtime where cuda:0 is absent."""
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu', weights_only=False)
        return super().find_class(module, name)

# Feature layout of the 1152-d SHAP input, read off the upstream DeepSHAP notebook
# (docs/provenance/upstream_SHAP_analysis.ipynb, cell 4/5):
#   x_feature_names = ['text_0..1023'] + ['video_0..63'] + ['audio_0..63']
MODALITY_SLICES = {'text': slice(0, 1024), 'video': slice(1024, 1088), 'audio': slice(1088, 1152)}

SHAP_TEST_FEATURE = {}   # kept for the Week 1 provenance cross-check
day0_modality = {}

for model_name, pkl_path in SHAP_PKLS.items():
    with open(pkl_path, 'rb') as f:
        shap_data = _CPUUnpickler(f).load()
    per_class = [np.abs(np.asarray(a)) for a in shap_data['SHAP_value']]
    assert len(per_class) == len(EMOTION_CLASSES), \
        f"Expected {len(EMOTION_CLASSES)} per-class SHAP arrays, got {len(per_class)}"
    stacked = np.stack(per_class, axis=0)  # (6 classes, N samples, 1152 dims)
    assert stacked.shape[2] == 1152, f"Unexpected SHAP feature width {stacked.shape[2]}"
    assert stacked.shape[1] == len(test_ids), (
        f"HALT: SHAP pickle {os.path.basename(pkl_path)} covers {stacked.shape[1]} samples but the "
        f"RML test split has {len(test_ids)}. Day 0 step 2 must be computed on the same samples the "
        "fidelity gate just evaluated."
    )
    SHAP_TEST_FEATURE[model_name] = torch.as_tensor(np.asarray(shap_data['test_feature']))

    mass = {m: float(stacked[:, :, s].sum(axis=2).mean()) for m, s in MODALITY_SLICES.items()}
    per_neuron = {m: float(stacked[:, :, s].mean(axis=2).mean()) for m, s in MODALITY_SLICES.items()}
    per_class_winner = {}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        row = {m: float(stacked[class_idx][:, s].mean()) for m, s in MODALITY_SLICES.items()}
        per_class_winner[class_name] = max(row, key=row.get)

    total_mass = sum(mass.values())
    ranked = sorted(per_neuron.items(), key=lambda kv: -kv[1])
    margin = ranked[0][1] / ranked[1][1]

    print(f"\n=== Day 0 step 2 — {model_name} model (N={stacked.shape[1]} test samples, 6 classes) ===")
    print(f"{'':<26}{'text':>12}{'video':>12}{'audio':>12}   rank-1")
    print(f"{'aggregate mass sum|phi|':<26}{mass['text']:>12.6f}{mass['video']:>12.6f}"
          f"{mass['audio']:>12.6f}   {max(mass, key=mass.get)}")
    print(f"{'per-neuron  mean|phi|':<26}{per_neuron['text']:>12.6f}{per_neuron['video']:>12.6f}"
          f"{per_neuron['audio']:>12.6f}   {ranked[0][0]}")
    print(f"  per-neuron ranking: {ranked[0][0]} > {ranked[1][0]} > {ranked[2][0]} "
          f"({ranked[0][0]} is {margin:.2f}x {ranked[1][0]})")
    print(f"  audio share of total attribution mass: {mass['audio'] / total_mass * 100:.1f}%")
    print(f"  per-class per-neuron winners: {per_class_winner}")

    day0_modality[model_name] = {
        'n_samples': int(stacked.shape[1]),
        'aggregate_mass_sum_abs_phi': mass,
        'per_neuron_mean_abs_phi': per_neuron,
        'per_neuron_ranking': [m for m, _ in ranked],
        'per_neuron_margin_over_runner_up': margin,
        'audio_share_of_total_mass': mass['audio'] / total_mass,
        'per_class_per_neuron_winner': per_class_winner,
    }

# ---------------------------------------------------------------------------
# Verdict.
#
# ADR 0001 Decision 1 specifies the dimension-invariant metric and anticipated a
# Text/Audio near-tie needing a 5% tie-break. Recomputed, it is not a tie: audio
# is rank-1 per-neuron for every class in both models by a wide margin, while
# text leads on aggregate mass purely because it has 16x more dimensions. Both
# statements are the same measurement, and together they reproduce the thesis's
# own wording -- "In the RML model, the dominant modality is text and audio"
# (SS4.4) -- with video a distant third on both views.
#
# This does not "resolve a tie in audio's favour"; it removes the tie. The write-up
# must say per-neuron vs aggregate, never "two metrics disagree" -- see ADR 0005.
# ---------------------------------------------------------------------------
DOMINANT_MODALITY = 'audio'

for model_name, res in day0_modality.items():
    assert res['per_neuron_ranking'][0] == DOMINANT_MODALITY, (
        f"HALT: Day 0 step 2 ranks '{res['per_neuron_ranking'][0]}', not '{DOMINANT_MODALITY}', "
        f"first on per-neuron attribution for the {model_name} model. Every cell below hooks "
        "model.a_transformer (the audio CLS). Do not proceed -- revisit ADR 0001 Decision 1 "
        "and ADR 0005 before ablating anything."
    )
    off = [c for c, w in res['per_class_per_neuron_winner'].items() if w != DOMINANT_MODALITY]
    assert not off, (
        f"HALT: Day 0 step 2 per-neuron winner is not '{DOMINANT_MODALITY}' for {off} in the "
        f"{model_name} model. The single-modality framing does not hold uniformly; stop and "
        "decide how Section VI-D should handle the exceptions before sweeping."
    )

with open(os.path.join(project_path, 'results', 'day0_dominant_modality.json'), 'w') as f:
    json.dump({'dominant_modality': DOMINANT_MODALITY, 'per_model': day0_modality}, f, indent=2, default=float)

print(f"\nDay 0 step 2 PASSED: dominant modality = {DOMINANT_MODALITY} (per-neuron, unanimous across "
      f"{len(EMOTION_CLASSES)} classes x 2 models). Saved results/day0_dominant_modality.json")

# Day 0 (cont.) — Falsification Pair

Before any real ablation sweep, two controls on the base model: **(a)** ablate all 64 audio dimensions — accuracy must move substantially; **(b)** ablate 5 random dimensions — the drop must be near zero.

Plus a third, threshold-free check that runs first: under a full 64-dim clamp every sample must end up with an *identical* `a_cls`. That is mechanical proof the hook writes into the tensor reaching `a_out`, so a small accuracy movement in (a) can be read as "audio isn't load-bearing" rather than "the hook silently did nothing" — the ambiguity that made v1/v2's numbers impossible to interpret.

This is a harness sanity check, not a scientific result. If it fails, Week 2 must not run.

In [ ]:
# -- Day 0: Falsification Pair (harness sanity check, not a scientific result) --

class MeanAblationHook:
    """Overwrites `target_indices` of the hooked module's output with `mean_vector`.

    Registered on model.a_transformer, whose forward (called with get_cls=True)
    returns the 2D [B, 64] CLS-token representation that feeds directly into
    a_out and the fusion sum (confirmed in journal 2026-08-09 / ADR 0004 --
    there is no separate FFN layer to distinguish this from).
    """
    def __init__(self, target_indices, mean_vector):
        self.target_indices = list(target_indices)
        self.mean_vector = mean_vector

    def __call__(self, module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        if not self.target_indices:
            return output
        modified = out.clone()
        clamp = self.mean_vector.to(device=modified.device, dtype=modified.dtype)
        modified[:, self.target_indices] = clamp[self.target_indices]
        if isinstance(output, tuple):
            return (modified,) + output[1:]
        return modified

def _capture_hook(sink):
    def fn(module, input, output):
        out = output[0] if isinstance(output, tuple) else output
        sink.append(out.detach().cpu())
    return fn

def capture_a_transformer_mean(model, loader):
    """Mean per-dimension activation of model.a_transformer's output over `loader`.
    Used only for this sanity check; Week 2's real ablation mean is computed
    over the training split, not the test split (see Day 6-7 cell)."""
    acts = []
    handle = model.a_transformer.register_forward_hook(_capture_hook(acts))
    evaluate_accuracy(model, loader)  # discard accuracy, just want the hook to fire
    handle.remove()
    return torch.cat(acts, dim=0).mean(dim=0)

def evaluate_with_ablation(model, loader, target_indices, mean_vector, capture=False):
    """Returns accuracy, and (if capture) the post-ablation a_transformer output.

    PyTorch feeds each forward hook the output as returned by the previous hook,
    so registering the capture hook second means it observes the already-clamped
    tensor -- which is exactly what the mechanical check below needs.
    """
    sink = []
    handles = [model.a_transformer.register_forward_hook(
        MeanAblationHook(target_indices, mean_vector))]
    if capture:
        handles.append(model.a_transformer.register_forward_hook(_capture_hook(sink)))
    acc = evaluate_accuracy(model, loader)
    for h in handles:
        h.remove()
    return (acc, torch.cat(sink, dim=0)) if capture else acc

print("Computing test-set mean activation of model.a_transformer for the falsification pair...")
falsification_mean = capture_a_transformer_mean(base_model, test_loader)

full_knockout_acc, clamped_acts = evaluate_with_ablation(
    base_model, test_loader, list(range(64)), falsification_mean, capture=True)

rng = np.random.default_rng(0)
random_5 = rng.choice(64, size=5, replace=False).tolist()
random_5_acc = evaluate_with_ablation(base_model, test_loader, random_5, falsification_mean)

full_knockout_drop = base_acc - full_knockout_acc
random_5_drop = base_acc - random_5_acc

# ---------------------------------------------------------------------------
# Check 0 -- mechanical, threshold-free proof that the hook actually writes.
#
# With all 64 dims clamped, every sample's a_cls is the SAME vector, so
# a_out(a_cls) contributes an identical constant to every sample's logits.
# If this holds, a small accuracy movement below means "audio isn't very
# load-bearing", NOT "the hook silently did nothing" -- which is the ambiguity
# that made v1/v2's numbers impossible to interpret.
# ---------------------------------------------------------------------------
assert clamped_acts.shape[1] == 64, f"Unexpected a_transformer output width {clamped_acts.shape[1]}"
max_spread = (clamped_acts - clamped_acts[0:1]).abs().max().item()
assert max_spread < 1e-5, (
    f"HALT: under a full 64-dim clamp the a_transformer output still varies across samples "
    f"(max deviation {max_spread:.3e}). The ablation hook is NOT writing into the tensor that "
    "reaches a_out. Nothing below this cell is interpretable until that is fixed."
)
print(f"\nHook write-through verified: under a full 64-dim clamp all {clamped_acts.shape[0]} "
      f"samples share one identical a_cls (max deviation {max_spread:.2e}).")

print(f"\n=== Falsification Pair (base model, unablated = {base_acc:.2f}%) ===")
print(f"Full 64-dim audio knockout: {full_knockout_acc:.2f}%  (drop = {full_knockout_drop:+.2f} pts)")
print(f"Random 5-dim control:       {random_5_acc:.2f}%  (drop = {random_5_drop:+.2f} pts, indices={random_5})")

assert full_knockout_drop > 10.0, (
    f"HALT: full audio knockout only moved accuracy by {full_knockout_drop:.2f} points, even though "
    "the write-through check above passed, so the hook IS firing. That means audio carries less "
    "causal weight than Day 0 step 2's attribution implies -- a scientific finding, not a bug, and one "
    "that changes what Section VI-D can claim. Stop and reconcile it with Day 0 step 2 before sweeping."
)
assert random_5_drop < 10.0, (
    f"HALT: a random 5-neuron ablation moved accuracy by {random_5_drop:.2f} points, "
    "comparable to a full 64-dim knockout. This means ablating almost any 5 neurons "
    "produces a large effect, which would make any single class's top-5 result "
    "uninterpretable as class-selective. Investigate before proceeding."
)
print("\nFalsification pair PASSED: the hook has a real, appropriately-scaled causal effect.")

# Week 1 — Days 1–2: Extract and Cache Activations

Protocol Days 1–2: forward pass over the full RML dataset for both models, caching the target layer's activation vector and the ground-truth label for every sample, then reloading to confirm shapes and identical ordering across models.

Cached per (model, split): the 64-d audio CLS (`model.a_transformer`'s output), the text and video logits, the label, and the retained utterance IDs. Every split is cached separately so probe fitting can use the train split alone — the 144-sample test split stays held out through Week 1.

Each cache carries a provenance stamp (checkpoint SHAs, meta path, split ID hashes). A resumed run that finds a stamp mismatch — or an unstamped cache from an older notebook — halts rather than mixing eras.

In [ ]:
# -- Week 1, Days 1-2: Extract and cache activations (train=518, valid=58, test=144) --
#
# One forward pass per (model, split) captures everything the rest of the notebook
# needs. Besides the 64-d audio CLS this also caches the text and video *logits*,
# because MME2E's head is exactly linear:
#
#   logits = weighted_fusion(stack([t_out(t_cls), v_out(v_cls), a_out(a_cls)], -1))
#
# with weighted_fusion = nn.Linear(3, 1, bias=False) and mod='tav'. Mean ablation
# only ever touches a_cls, so (t_logits, v_logits, a_cls) is a lossless summary
# for every ablation this notebook runs -- see the Week 2 fast-path cell.

ACTS_DIR = os.path.join(CHECKPOINTS_DIR, 'activations')
CACHE_VERSION = 3  # bump whenever the cached tensor set changes shape/meaning

def extract_activations(model, loader):
    model.eval()
    a_cls, t_logits, v_logits, labels, uttr_ids = [], [], [], [], []
    store = {}
    def sink(key):
        def fn(module, input, output):
            out = output[0] if isinstance(output, tuple) else output
            store[key] = out.detach().cpu()
        return fn
    handles = [
        model.a_transformer.register_forward_hook(sink('a')),
        model.t_out.register_forward_hook(sink('t')),
        model.v_out.register_forward_hook(sink('v')),
    ]
    try:
        with torch.no_grad():
            for batch in tqdm(loader, desc="Extracting activations", leave=False):
                ids, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
                text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                         max_length=MODEL_ARGS['text_max_len'],
                                         padding='max_length', truncation=True)
                text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
                imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
                specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
                _ = model(imgs, img_lens, specs, spec_lens, text_inputs)
                a_cls.append(store['a']); t_logits.append(store['t']); v_logits.append(store['v'])
                labels.extend(Y.argmax(-1).cpu().numpy())
                # collate_fn drops any sample with zero sampled frames, so the
                # retained IDs are the ground truth for what was actually seen --
                # never the split file. Day 0's integrity gate makes drops
                # impossible, and this is how we prove none happened.
                uttr_ids.extend(list(ids))
    finally:
        for h in handles:
            h.remove()
    return (torch.cat(a_cls, dim=0), torch.cat(t_logits, dim=0), torch.cat(v_logits, dim=0),
            torch.tensor(labels, dtype=torch.long), uttr_ids)

train_loader = build_loader(train_ids)
valid_loader = build_loader(valid_ids)
# test_loader already built during the Day 0 fidelity gate.

SPLIT_IDS = {'train': train_ids, 'valid': valid_ids, 'test': test_ids}
LOADERS = {'train': train_loader, 'valid': valid_loader, 'test': test_loader}

# Provenance stamp so a resumed run can't silently reuse activations cached
# against a different checkpoint/data source (the exact failure mode that
# forced the v3 restart -- see ADR 0004). A cache directory left over from a
# prior notebook version has no stamp file and must not be trusted implicitly.
CACHE_PROVENANCE = {
    'cache_version': CACHE_VERSION,
    'base_ckpt_sha256': compute_sha256(BASE_CKPT),
    'ft_ckpt_sha256': compute_sha256(FT_CKPT),
    'meta_path': META_PATH,
    'split_sizes': {k: len(v) for k, v in SPLIT_IDS.items()},
    'split_id_sha256': {k: hashlib.sha256('\n'.join(v).encode()).hexdigest()
                        for k, v in SPLIT_IDS.items()},
}

TENSOR_NAMES = ['acts', 'tlogits', 'vlogits', 'labels']

def _cache_paths(model_name, split_name):
    paths = {n: os.path.join(ACTS_DIR, f'{model_name}_{split_name}_{n}.pt') for n in TENSOR_NAMES}
    paths['ids'] = os.path.join(ACTS_DIR, f'{model_name}_{split_name}_ids.json')
    paths['stamp'] = os.path.join(ACTS_DIR, f'{model_name}_{split_name}_provenance.json')
    return paths

activations = {}  # activations[model_name][split] = dict(acts, tlogits, vlogits, labels, ids)
for model_name, model in [('base', base_model), ('finetuned', ft_model)]:
    activations[model_name] = {}
    for split_name, loader in LOADERS.items():
        p = _cache_paths(model_name, split_name)
        cache_present = all(os.path.exists(p[n]) for n in TENSOR_NAMES) and os.path.exists(p['ids'])
        if cache_present and not os.path.exists(p['stamp']):
            raise RuntimeError(
                f"HALT: cached tensors for {model_name}/{split_name} exist but have no provenance "
                "stamp -- they predate this notebook's provenance check and may have been "
                "extracted against a retired checkpoint/data source (see ADR 0004). Delete "
                "checkpoints/activations/ and re-run this cell rather than trusting them."
            )
        if cache_present:
            with open(p['stamp']) as f:
                stamp = json.load(f)
            if stamp != CACHE_PROVENANCE:
                raise RuntimeError(
                    f"HALT: cached {model_name}/{split_name} provenance does not match this run.\n"
                    f"  cached:  {stamp}\n  current: {CACHE_PROVENANCE}\n"
                    "Delete checkpoints/activations/ and re-run this cell rather than mixing eras."
                )
            rec = {n: torch.load(p[n]) for n in TENSOR_NAMES}
            with open(p['ids']) as f:
                rec['ids'] = json.load(f)
            print(f"Resume gate: loaded cached {model_name}/{split_name} "
                  f"({rec['acts'].shape[0]} samples), provenance verified.")
        else:
            print(f"Extracting {model_name}/{split_name}...")
            acts, tl, vl, labels, ids = extract_activations(model, loader)
            rec = {'acts': acts, 'tlogits': tl, 'vlogits': vl, 'labels': labels, 'ids': ids}
            for n in TENSOR_NAMES:
                torch.save(rec[n], p[n])
            with open(p['ids'], 'w') as f:
                json.dump(ids, f)
            with open(p['stamp'], 'w') as f:
                json.dump(CACHE_PROVENANCE, f, indent=2)
        activations[model_name][split_name] = rec

# --- Consistency checks (assert against what was retained, not against literals) ---
for split_name in LOADERS:
    b, ft = activations['base'][split_name], activations['finetuned'][split_name]
    assert b['ids'] == ft['ids'], (
        f"HALT: base and fine-tuned runs retained different samples for '{split_name}'. "
        "Every cross-model comparison below assumes row i is the same utterance in both."
    )
    assert torch.equal(b['labels'], ft['labels']), \
        f"HALT: base/fine-tuned label vectors differ for '{split_name}'."
    dropped = [uid for uid in SPLIT_IDS[split_name] if uid not in set(b['ids'])]
    assert not dropped, (
        f"HALT: collate_fn dropped {len(dropped)} zero-frame sample(s) from '{split_name}' "
        f"(e.g. {dropped[:5]}). Day 0's integrity gate should have caught this first -- "
        "re-run Day 0 and read its output before continuing."
    )
    assert b['ids'] == SPLIT_IDS[split_name], (
        f"HALT: retained '{split_name}' IDs are not the split file's order. The SHAP "
        "cross-check and every per-sample join below assume split-file order."
    )
    for model_name in activations:
        rec = activations[model_name][split_name]
        n = len(SPLIT_IDS[split_name])
        assert rec['acts'].shape == (n, 64), \
            f"{model_name}/{split_name}: a_cls expected ({n}, 64), got {tuple(rec['acts'].shape)}"
        assert rec['tlogits'].shape == (n, len(EMOTION_CLASSES)), \
            f"{model_name}/{split_name}: t_logits shape {tuple(rec['tlogits'].shape)}"
        assert rec['vlogits'].shape == (n, len(EMOTION_CLASSES)), \
            f"{model_name}/{split_name}: v_logits shape {tuple(rec['vlogits'].shape)}"

# --- Day 0 step 2 provenance cross-check ---
# The SHAP pickles' `test_feature` is [text_cls 1024 | v_cls 64 | a_cls 64] over the
# 144 test samples. Columns 1088: are therefore the very tensor just extracted. If
# they agree, the SHAP pickle is provably computed from THIS checkpoint on THESE
# samples -- upgrading the Day 0 step 2 provenance from "the filename says 0.6724" to a
# measurement. Correlation, not allclose: the upstream run used a different GPU,
# torch build and batch size, so exact float equality is not expected.
print("\n=== Day 0 step 2 cross-check: SHAP test_feature[:, 1088:] vs. extracted test a_cls ===")
for model_name in ['base', 'finetuned']:
    ours = activations[model_name]['test']['acts'].float().numpy().ravel()
    theirs = SHAP_TEST_FEATURE[model_name][:, 1088:1152].float().numpy().ravel()
    corr = float(np.corrcoef(ours, theirs)[0, 1])
    max_abs = float(np.abs(ours - theirs).max())
    print(f"  {model_name:10s} pearson r = {corr:.6f}   max|diff| = {max_abs:.3e}")
    assert corr > 0.99, (
        f"HALT: the {model_name} SHAP pickle's audio block does not match this run's a_cls "
        f"(r = {corr:.4f}). Day 0 step 2 was computed on a different checkpoint, a different sample "
        "order, or a different layer -- its audio verdict cannot be carried into Week 2. "
        "Stop and re-derive Day 0 step 2 before ablating."
    )
print("  Day 0 step 2 provenance confirmed: SHAP attributions describe the tensor we ablate.")

print("\nWeek 1 Days 1-2 complete. Cached shapes:")
for model_name in activations:
    for split_name, rec in activations[model_name].items():
        print(f"  {model_name:10s} {split_name:6s} a_cls={tuple(rec['acts'].shape)} "
              f"t_logits={tuple(rec['tlogits'].shape)} v_logits={tuple(rec['vlogits'].shape)} "
              f"labels={tuple(rec['labels'].shape)}")

# Week 1 — Day 3: Fit Sparse L1-Penalized Probes (train split only)

One binary L1-logistic probe per (model, class), fit strictly on the 518-sample train split. Ranks the 64 dimensions by absolute weight magnitude, giving a per-class, per-model candidate neuron ranking. Never fit on valid or test.

In [ ]:
# -- Week 1, Day 3: Fit L1 probes on the train split, rank neurons per class --
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def fit_class_probes(train_acts, train_labels):
    """Returns {class_name: (scaler, probe, ranked_indices, ranked_weights)}."""
    X_train = train_acts.numpy()
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    probes = {}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        y_bin = (train_labels.numpy() == class_idx).astype(int)
        probe = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=2000)
        probe.fit(X_train_scaled, y_bin)
        weights = probe.coef_[0]
        # Ties in |weight| (common with L1, which zeroes many dims outright) are
        # broken by index so the ranking is deterministic across runs.
        ranked_indices = np.lexsort((np.arange(len(weights)), -np.abs(weights)))
        probes[class_name] = {
            'scaler': scaler,
            'probe': probe,
            'ranked_indices': ranked_indices.tolist(),
            'ranked_weights': weights[ranked_indices].tolist(),
            'n_nonzero_weights': int((weights != 0).sum()),
        }
    return probes

class_probes = {}
for model_name in ['base', 'finetuned']:
    rec = activations[model_name]['train']
    class_probes[model_name] = fit_class_probes(rec['acts'], rec['labels'])
    print(f"\n=== {model_name.title()} model: top-5 neurons per class (train-fit, N=518) ===")
    for class_name in EMOTION_CLASSES:
        info = class_probes[model_name][class_name]
        top5 = [(idx, round(w, 3)) for idx, w in zip(info['ranked_indices'][:5], info['ranked_weights'][:5])]
        print(f"  {class_name:10s} nonzero_L1_weights={info['n_nonzero_weights']:2d}/64  {top5}")
        # Protocol Day 3 ranks the top-k by |weight|. If L1 zeroed more than 59 of
        # the 64 dims, the "top 5" would contain zero-weight dims chosen only by
        # index order -- a ranking with no signal in it.
        assert info['n_nonzero_weights'] >= 5, (
            f"HALT: L1 left only {info['n_nonzero_weights']} nonzero weights for "
            f"{model_name}/{class_name}; the top-5 ranking would be arbitrary. Loosen the "
            "penalty (raise C) and record the change before continuing."
        )

# Week 1 — Day 4: Sanity-Check the Probes

ROC-AUC on the 144-sample test split, held out from Day 3's fit. Protocol fallback trigger: mean AUC < 0.65 across the six classes, or more than 2 classes below 0.55.

In [ ]:
# -- Week 1, Day 4: Sanity-check the probes (held-out test AUC) --
#
# Protocol Day 4 allows same-data evaluation at this exploratory stage; this
# uses the held-out 144-sample test split instead, which is strictly stronger
# and keeps Week 1 leakage-free (the probes were fit on the 518 train samples only).

probe_auc = {}
for model_name in ['base', 'finetuned']:
    rec = activations[model_name]['test']
    X_test = rec['acts'].numpy()
    y_test = rec['labels'].numpy()
    print(f"\n=== {model_name.title()} model: held-out test AUC (N_test={len(y_test)}) ===")
    aucs = []
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        info = class_probes[model_name][class_name]
        X_test_scaled = info['scaler'].transform(X_test)
        y_bin = (y_test == class_idx).astype(int)
        scores = info['probe'].predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_bin, scores)
        aucs.append(auc)
        print(f"  {class_name:10s} AUC = {auc:.4f}")
    mean_auc = float(np.mean(aucs))
    n_below_055 = sum(a < 0.55 for a in aucs)
    print(f"  Mean AUC = {mean_auc:.4f}  |  classes with AUC < 0.55: {n_below_055}")
    probe_auc[model_name] = {'per_class': dict(zip(EMOTION_CLASSES, aucs)), 'mean': mean_auc}
    if mean_auc < 0.65 or n_below_055 > 2:
        # Protocol Day 4's remedy is "go back to Day 0 step 3 and try the CLS-token
        # layer instead of the FFN output, or vice versa". ADR 0004 established
        # that for MME2E there is no separate FFN output: a_transformer's get_cls
        # return IS the tensor feeding a_out and the fusion sum, so that fallback
        # has no second rung. If this fires, the real options are a different
        # modality's CLS (which contradicts Day 0 step 2) or a different layer
        # depth -- either is a methodology change needing a new ADR, not a rerun.
        print(f"  WARNING: {model_name} fails the Day 4 threshold (mean AUC < 0.65 or >2 classes "
              "< 0.55). The audio CLS may not carry a clean linear class signal. Per ADR 0004 the "
              "FFN-vs-CLS fallback does not exist for this architecture -- stop and write an ADR "
              "before choosing a different target.")

with open(os.path.join(project_path, 'results', 'week1_probe_auc.json'), 'w') as f:
    json.dump(probe_auc, f, indent=2, default=float)
print("\nSaved results/week1_probe_auc.json")

# Week 2 — Days 6–7: Mean-Ablation Harness (exact fast path)

Protocol Days 6–7: *"Use mean ablation, not zero ablation... computed once, upfront, over the training set... Implement as a forward hook."* The hook (`MeanAblationHook`) is already defined and proven in Day 0's falsification pair; the train-set mean vector is computed here.

**Why there is a fast path.** With `mod='tav'`, `MME2E.forward` ends in

```
logits = weighted_fusion(stack([t_out(t_cls), v_out(v_cls), a_out(a_cls)], -1)).squeeze(-1)
```

and `weighted_fusion` is `nn.Linear(3, 1, bias=False)`. Mean ablation touches **only** `a_cls`. So the `(t_logits, v_logits, a_cls)` triple cached in Days 1–2 determines the model's output under *any* ablation of the audio CLS, exactly — this is algebra, not an approximation.

Days 8–9 run 6 classes × 8 values of k × 2 models, plus Day 14's transfer, plus baselines: over 100 full evaluations. On the slow path each one re-reads 144 wavs and ~1,300 JPEGs across the Colab Drive mount — roughly 150,000 file reads, which is where a long run actually dies (the original training log shows test eval at ~9 it/s, so GPU time was never the constraint). On the fast path each becomes one 64×6 matmul.

The cell **proves** the fast path before using it: it reproduces the unablated baseline, the full-64 knockout, and one class at k=5 through real hooked forward passes and compares. If any check disagrees, it prints loudly and falls back to the slow path — the slow path is the reference, so falling back is always safe, just slower.

In [ ]:
# -- Week 2, Days 6-7: Train-set mean ablation vector + exact fast-path harness --

# Protocol Day 6-7: the mean is computed ONCE, over the training split, never
# over the split being evaluated. (Day 0's falsification pair used a test-set
# mean deliberately -- it was a harness check, not a scientific measurement.)
train_mean = {}
for model_name in ['base', 'finetuned']:
    train_acts = activations[model_name]['train']['acts']
    train_mean[model_name] = train_acts.mean(dim=0)
    print(f"{model_name.title()} train-set mean vector: shape={tuple(train_mean[model_name].shape)}, "
          f"mean={train_mean[model_name].mean().item():.4f}, std={train_mean[model_name].std().item():.4f}")

def per_class_accuracies(preds, targets):
    accs = {'overall': float(np.mean(preds == targets)) * 100.0}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        mask = targets == class_idx
        accs[class_name] = float(np.mean(preds[mask] == targets[mask])) * 100.0 if mask.sum() > 0 else float('nan')
    return accs

def evaluate_per_class(model, loader):
    """Slow path: a real forward pass over `loader`. The reference implementation."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            _, imgs, img_lens, specs, spec_lens, text_batch, Y = batch
            text_inputs = tokenizer(list(text_batch), return_tensors='pt',
                                     max_length=MODEL_ARGS['text_max_len'],
                                     padding='max_length', truncation=True)
            text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
            imgs = imgs.to(device) if hasattr(imgs, 'to') else torch.tensor(imgs, device=device)
            specs = specs.to(device) if hasattr(specs, 'to') else torch.tensor(specs, device=device)
            logits = model(imgs, img_lens, specs, spec_lens, text_inputs)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_targets.extend(Y.argmax(-1).cpu().numpy())
    return per_class_accuracies(np.array(all_preds), np.array(all_targets))

def evaluate_per_class_slow_ablated(model, loader, target_indices, mean_vector):
    handle = model.a_transformer.register_forward_hook(
        MeanAblationHook(target_indices, mean_vector))
    try:
        return evaluate_per_class(model, loader)
    finally:
        handle.remove()

# --- Fast path: replay the linear head over cached (t_logits, v_logits, a_cls) ---
HEAD = {}
for model_name, model in [('base', base_model), ('finetuned', ft_model)]:
    HEAD[model_name] = {
        'a_out_w': model.a_out.weight.detach().cpu().clone(),           # (6, 64)
        'a_out_b': model.a_out.bias.detach().cpu().clone(),             # (6,)
        'fusion_w': model.weighted_fusion.weight.detach().cpu().clone() # (1, 3) over [t, v, a]
    }
    assert HEAD[model_name]['fusion_w'].shape == (1, 3), (
        "HALT: weighted_fusion is not the expected Linear(3, 1) over [t, v, a]. The fast path's "
        "algebra assumes MME2E.forward's 'tav' branch; re-read src/models/e2e.py before trusting it."
    )

EVAL_SPLIT = 'test'  # Days 8-9 evaluate on the held-out 144-sample test split

def fast_per_class(model_name, target_indices=(), mean_vector=None, split=EVAL_SPLIT):
    rec = activations[model_name][split]
    a = rec['acts'].clone()
    idx = list(target_indices)
    if idx:
        clamp = mean_vector.to(a.dtype)
        a[:, idx] = clamp[idx]
    h = HEAD[model_name]
    a_logits = a @ h['a_out_w'].T + h['a_out_b']                      # (N, 6)
    stacked = torch.stack([rec['tlogits'], rec['vlogits'], a_logits], dim=-1)  # (N, 6, 3)
    logits = (stacked * h['fusion_w'].view(1, 1, 3)).sum(dim=-1)      # (N, 6)
    return per_class_accuracies(logits.argmax(-1).numpy(), rec['labels'].numpy())

# --- Discriminator: prove the fast path against real forward passes ---
# Three checks, chosen to exercise every branch the sweep will use: no ablation,
# the maximal ablation, and a realistic top-k ablation.
print("\nVerifying the fast path against real hooked forward passes (3 checks)...")
# Pick the class whose top-5 carries the most probe weight, not simply the first
# class. L1 zeroes many dimensions outright, and the Day 3 tie-break then fills a
# short top-5 by index order. Testing such a class would clamp dimensions the model
# barely uses, so `delta < 1e-6` would pass trivially and prove nothing.
_probe_class = max(
    EMOTION_CLASSES,
    key=lambda c: float(np.abs(class_probes['base'][c]['ranked_weights'][:5]).sum()))
_probe_top5 = class_probes['base'][_probe_class]['ranked_indices'][:5]
_checks = [
    ('unablated baseline', [], None),
    ('full 64-dim knockout', list(range(64)), train_mean['base']),
    (f'{_probe_class} top-5 (k=5)', _probe_top5, train_mean['base']),
]
USE_FAST_PATH = True
_fast_verification = {}
for label, idx, mv in _checks:
    slow = (evaluate_per_class(base_model, test_loader) if not idx
            else evaluate_per_class_slow_ablated(base_model, test_loader, idx, mv))
    fast = fast_per_class('base', idx, mv)
    delta = max(abs(slow[k] - fast[k]) for k in slow if not math.isnan(slow[k]))
    _fast_verification[label] = {'slow_overall': slow['overall'], 'fast_overall': fast['overall'],
                                 'max_per_class_delta_pp': delta}
    ok = delta < 1e-6
    print(f"  {label:26s} slow={slow['overall']:6.2f}%  fast={fast['overall']:6.2f}%  "
          f"max per-class delta={delta:.2e}pp  {'OK' if ok else 'MISMATCH'}")
    if not ok:
        USE_FAST_PATH = False

if USE_FAST_PATH:
    print("\nFast path VERIFIED -- it reproduces the forward pass exactly. Days 8-9 and Day 14 "
          "will use it (~100 evaluations become matmuls instead of ~150k Drive file reads).")
else:
    print("\n*** FAST PATH REJECTED: it did not reproduce the forward pass. Falling back to real "
          "forward passes for every evaluation. This is SAFE (the slow path is the reference) but "
          "will take substantially longer -- keep the Colab tab alive. Record this in the journal. ***")

def evaluate_ablated(model_name, model, loader, target_indices, mean_vector):
    """Single entry point for Days 8-9 and Day 14. Identical results either way."""
    if USE_FAST_PATH:
        return fast_per_class(model_name, target_indices, mean_vector)
    if not list(target_indices):
        return evaluate_per_class(model, loader)
    return evaluate_per_class_slow_ablated(model, loader, target_indices, mean_vector)

# Early drift check, before the sweep rather than after it. On a resumed run the
# cached t/v logits come off disk; the provenance stamp proves they were extracted
# under these checkpoint SHAs, and this proves the whole cached evaluation path
# still reproduces Day 0's fidelity gate. Same comparison the sweep repeats at the
# end -- run here it costs two evaluations instead of a whole sweep.
for _mn, _model, _gate in [('base', base_model, base_acc), ('finetuned', ft_model, ft_acc)]:
    _cached = evaluate_ablated(_mn, _model, test_loader, [], None)['overall']
    assert abs(_cached - _gate) < 1e-6, (
        f"HALT: the cached-activation evaluation path gives {_cached:.4f}% for the {_mn} model "
        f"but Day 0's fidelity gate measured {_gate:.4f}%. The cached tensors and the live model "
        "have drifted apart. Delete checkpoints/activations/ and re-run Week 1 Days 1-2."
    )
print("Cached evaluation path reproduces the Day 0 fidelity gate for both models.")

with open(os.path.join(project_path, 'results', 'week2_fast_path_verification.json'), 'w') as f:
    json.dump({'use_fast_path': USE_FAST_PATH, 'checks': _fast_verification}, f, indent=2, default=float)
print("Saved results/week2_fast_path_verification.json")

# Week 2 — Days 8–9: Causal Ablation Sweep

Protocol Day 8 (base model) and Day 9 (fine-tuned model, using its *own* Day 3 rankings). For each class, ablate its top-k neurons and record per-class accuracy for **all** classes — the selectivity, not the raw drop, is the causal evidence.

k = 1, 3, 5, 10 per protocol, extended with 16/32/48/64 for a dose-response curve. k=5 is the primary table entry. A target-class drop ≥ 2.5× the mean absolute non-target drop (ADR 0001 Decision 3) marks a causally class-selective feature set.

Evaluated on the held-out 144-sample test split — the same split Day 0's fidelity gate and falsification pair used.

In [ ]:
# -- Week 2, Days 8-9: Causal ablation sweep (base model, then fine-tuned model) --
#
# Protocol Day 8: for each class, ablate its top-k neurons (k = 1, 3, 5, 10 --
# 16/32/48/64 added here for a dose-response curve), run the full eval set, and
# record per-class accuracy for ALL classes, not just the target.
# Protocol Day 9: same procedure for the fine-tuned model, using the fine-tuned
# model's OWN Day 3 rankings -- not the base model's. (Cross-model ablation is
# Day 14's separate test.)

K_VALUES = [1, 3, 5, 10, 16, 32, 48, 64]

def run_ablation_sweep(model_name, model, loader):
    baseline = evaluate_ablated(model_name, model, loader, [], None)
    print(f"{model_name.title()} unablated baseline: overall={baseline['overall']:.2f}%")
    sweep = {'baseline': baseline}
    for class_name in tqdm(EMOTION_CLASSES, desc=f"{model_name} sweep"):
        ranked = class_probes[model_name][class_name]['ranked_indices']
        class_results = {}
        for k in K_VALUES:
            target_idx = ranked[:k]
            ablated = evaluate_ablated(model_name, model, loader, target_idx, train_mean[model_name])
            target_drop = baseline[class_name] - ablated[class_name]
            non_target_drops = [baseline[c] - ablated[c] for c in EMOTION_CLASSES if c != class_name]
            mean_non_target_drop = float(np.mean(non_target_drops))
            # ADR 0001 defines selectivity against the mean ABSOLUTE non-target drop.
            # The signed mean is reported too: a negative signed mean with a large
            # absolute mean means non-target classes moved in both directions, which
            # is polysemantic collateral damage rather than clean class-selectivity.
            mean_abs_non_target_drop = float(np.mean(np.abs(non_target_drops)))
            denom = max(mean_abs_non_target_drop, 1e-6)
            selectivity_ratio = target_drop / denom
            class_results[k] = {
                'ablated_accs': ablated,
                'target_indices': list(target_idx),
                'target_drop': target_drop,
                'mean_non_target_drop': mean_non_target_drop,
                'mean_abs_non_target_drop': mean_abs_non_target_drop,
                'selectivity_ratio': selectivity_ratio,
                'causally_class_selective': bool(selectivity_ratio >= 2.5),
            }
        sweep[class_name] = class_results
        r5 = class_results[5]
        print(f"  [{class_name:10s}] k=5 target_drop={r5['target_drop']:+.2f}pp  "
              f"mean|non-target| drop={r5['mean_abs_non_target_drop']:.2f}pp  "
              f"selectivity={r5['selectivity_ratio']:.2f}x  "
              f"{'SELECTIVE' if r5['causally_class_selective'] else '-'}")
    return sweep

ablation_sweeps = {}
for model_name, model in [('base', base_model), ('finetuned', ft_model)]:
    ablation_sweeps[model_name] = run_ablation_sweep(model_name, model, test_loader)

# Day 9 cross-check: the sweep's own unablated baseline must reproduce Day 0's
# fidelity-gate accuracy. If these ever diverge, the cached activations and the
# live model have drifted apart and nothing in Weeks 2-3 is trustworthy.
for model_name, gate_acc in [('base', base_acc), ('finetuned', ft_acc)]:
    sweep_acc = ablation_sweeps[model_name]['baseline']['overall']
    assert abs(sweep_acc - gate_acc) < 1e-6, (
        f"HALT: {model_name} sweep baseline {sweep_acc:.4f}% != Day 0 fidelity gate "
        f"{gate_acc:.4f}%. The evaluation path changed between Day 0 and Week 2."
    )
print("\nSweep baselines reproduce the Day 0 fidelity gate exactly.")

for model_name in ablation_sweeps:
    path = os.path.join(project_path, 'results', f'week2_{model_name}_ablation_sweep.json')
    with open(path, 'w') as f:
        json.dump(ablation_sweeps[model_name], f, indent=2, default=float)
    print(f"Saved {path}")

# Week 3 — Days 11–14: Does Fine-Tuning Preserve or Reassign the Neurons?

Protocol Days 11–12 build per-class selectivity vectors (`mean(act | class c) − mean(act | class ≠ c)`, train split, both models); Day 13 compares them by cosine similarity; Day 14 runs the decisive causal version — ablate the **base** model's top-5 neurons for each class inside the **fine-tuned** model.

Per protocol Day 14, if Day 13 and Day 14 disagree, **Day 14 is the headline result**.

In [ ]:
# -- Week 3, Days 11-13: Selectivity vectors + cosine similarity (train split) --
#
# Protocol Day 11-12: selectivity[d] = mean(act[d] | label = c) - mean(act[d] | label != c),
# computed the same way for both models. Day 13: cosine similarity per class.

def selectivity_vectors(train_acts, train_labels):
    X = train_acts.numpy()
    y = train_labels.numpy()
    vectors = {}
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        in_class = X[y == class_idx].mean(axis=0)
        out_class = X[y != class_idx].mean(axis=0)
        vectors[class_name] = in_class - out_class
    return vectors

selectivity = {}
for model_name in ['base', 'finetuned']:
    rec = activations[model_name]['train']
    selectivity[model_name] = selectivity_vectors(rec['acts'], rec['labels'])

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

cosine_similarities = {
    class_name: cosine_sim(selectivity['base'][class_name], selectivity['finetuned'][class_name])
    for class_name in EMOTION_CLASSES
}
print("=== Day 13: Base vs. Fine-Tuned selectivity vector cosine similarity ===")
for class_name, sim in cosine_similarities.items():
    print(f"  {class_name:10s} cos_sim = {sim:+.4f}")
print("\nPer protocol Day 14, if this disagrees with the ablation-transfer result below, "
      "the ablation-transfer result is the headline.")

In [ ]:
# -- Week 3, Day 14: Ablation transfer -- base model's top-5 neurons ablated INSIDE the fine-tuned model --
#
# Protocol Day 14: "take the base model's top-5 neurons for class c, and ablate
# those same indices in the fine-tuned model." The mean vector stays the
# fine-tuned model's own train-set mean -- we are relocating the *indices*, not
# importing the base model's activation statistics.

ft_baseline = ablation_sweeps['finetuned']['baseline']
transfer_results = {}
for class_name in EMOTION_CLASSES:
    base_top5 = class_probes['base'][class_name]['ranked_indices'][:5]
    ft_own_top5 = class_probes['finetuned'][class_name]['ranked_indices'][:5]
    ablated = evaluate_ablated('finetuned', ft_model, test_loader, base_top5, train_mean['finetuned'])
    ft_drop_using_base_neurons = ft_baseline[class_name] - ablated[class_name]
    transfer_results[class_name] = {
        'base_top5_indices': list(base_top5),
        'ft_own_top5_indices': list(ft_own_top5),
        'index_overlap': len(set(base_top5) & set(ft_own_top5)),
        'ft_drop_using_base_neurons': ft_drop_using_base_neurons,
        'ft_ablated_accs': ablated,
    }
    print(f"  {class_name:10s} base-top5={list(base_top5)}  overlap with FT's own top-5: "
          f"{transfer_results[class_name]['index_overlap']}/5  "
          f"FT drop: {ft_drop_using_base_neurons:+.2f}pp")

with open(os.path.join(project_path, 'results', 'week3_ablation_transfer.json'), 'w') as f:
    json.dump(transfer_results, f, indent=2, default=float)
print("Saved results/week3_ablation_transfer.json")

# Week 3 — Day 15: Compile the Transfer Retention Table

Retention Ratio `R = ft_drop / base_drop`, both from the same k=5 ablation, with `ft_drop` using the base model's neuron indices (Day 14). Per ADR 0004 the previous ε=0.05 "N/A" screen is **withdrawn** — it existed to hide artifacts from the v1/v2 broken harness. Raw R is reported for every class; small or negative `base_drop` is flagged, never suppressed.

In [ ]:
# -- Week 3, Day 15: Compile the Transfer Retention table --
#
# R = ft_drop / base_drop, both taken from the same k=5 ablation, with ft_drop
# using the BASE model's neuron indices (Day 14).
#
# Band definitions follow CONTEXT.md, which README names as the canonical
# terminology: Preservation R >= 0.80, Reassignment 0.20 <= R < 0.80,
# Dispersion R < 0.20. ADR 0001 Decision 4 states different numbers
# (>= 0.70 / < 0.30) and additionally splits Reassignment from Dispersion by
# whether the fine-tuned drop is "sparse" or "dense" -- a measure this protocol
# never defines or computes, and its bands leave 0.30 <= R < 0.70 unclassified.
# CONTEXT.md's bands are total and implementable, so they are what this cell
# uses; the conflict is recorded in ADR 0005. Raw R is reported for every class,
# so relabelling later costs nothing.
PRESERVATION_MIN = 0.80
DISPERSION_MAX = 0.20

# Per ADR 0004 the previous epsilon=0.05 "N/A" screen is WITHDRAWN -- it was written
# to hide artifacts from the v1/v2 broken harness, not a real statistical need.
# Small base_drop is flagged, never suppressed.
FLAG_THRESHOLD = 5.0  # pp; informational only

retention_table = []
for class_name in EMOTION_CLASSES:
    base_drop = ablation_sweeps['base'][class_name][5]['target_drop']
    ft_drop = transfer_results[class_name]['ft_drop_using_base_neurons']
    r = ft_drop / base_drop if abs(base_drop) > 1e-9 else float('nan')
    if np.isnan(r):
        outcome = 'Undefined (base_drop == 0)'
    elif r >= PRESERVATION_MIN:
        outcome = 'Substrate Preservation'
    elif r >= DISPERSION_MAX:
        outcome = 'Substrate Reassignment'
    else:
        outcome = 'Substrate Dispersion'
    retention_table.append({
        'class': class_name,
        'base_drop_pp': base_drop,
        'ft_drop_pp': ft_drop,
        'R': r,
        'outcome': outcome,
        'small_base_drop_flag': bool(base_drop < FLAG_THRESHOLD),
        'cosine_similarity': cosine_similarities[class_name],
        'base_ft_top5_overlap': transfer_results[class_name]['index_overlap'],
    })

import pandas as pd
retention_df = pd.DataFrame(retention_table)
print(retention_df.to_string(index=False))
retention_df.to_csv(os.path.join(project_path, 'results', 'week3_transfer_retention.csv'), index=False)
print("\nSaved results/week3_transfer_retention.csv")

flagged = retention_df[retention_df['small_base_drop_flag']]
if len(flagged) > 0:
    print(f"\nNote: {len(flagged)} class(es) have base_drop < {FLAG_THRESHOLD}pp. With 24 test "
          f"samples per class one sample is 4.17pp, so this flag means 'fewer than two samples "
          f"moved' -- read it as single-sample noise, not as a small but real effect. R for those "
          "classes is correspondingly sensitive to its denominator. Reported, not suppressed.")
negative = retention_df[retention_df['base_drop_pp'] < 0]
if len(negative) > 0:
    print(f"Note: {len(negative)} class(es) have a NEGATIVE base_drop (ablation improved that "
          "class's accuracy), which makes R's sign uninterpretable as a retention ratio. "
          "Read those rows from base_drop_pp / ft_drop_pp directly, not from R.")

# Week 4 — Days 16-18: Publication Tables & Dose-Response Figure

Table VII (base model, k=5 target vs. mean non-target drop per class), Table VIII (fine-tuned model, same), Table IX (cosine similarity + ablation-transfer drop per class, matching the columns in the IEEE paper's Section VI-D placeholder). Dose-response figure across k=1..64 for both models.

In [ ]:
# -- Week 4, Days 16-18: Export Tables VII/VIII/IX and the dose-response figure --
import matplotlib.pyplot as plt

def build_table_vii_viii(model_name):
    rows = []
    for class_name in EMOTION_CLASSES:
        r = ablation_sweeps[model_name][class_name][5]
        rows.append({
            'class': class_name,
            'top5_neurons': str(r['target_indices']),
            'target_class_drop_pp': r['target_drop'],
            'mean_non_target_drop_pp': r['mean_non_target_drop'],
            'mean_abs_non_target_drop_pp': r['mean_abs_non_target_drop'],
            'selectivity_ratio': r['selectivity_ratio'],
            'causally_class_selective': r['causally_class_selective'],
        })
    return pd.DataFrame(rows)

table_vii = build_table_vii_viii('base')       # Table VII: base model, k=5
table_viii = build_table_vii_viii('finetuned') # Table VIII: fine-tuned model, k=5
table_ix = retention_df[['class', 'cosine_similarity', 'ft_drop_pp', 'R', 'outcome']].rename(
    columns={'ft_drop_pp': 'delta_acc_target_class_base_neurons_in_ft'})

table_vii.to_csv(os.path.join(project_path, 'results', 'table_vii_base_ablation_k5.csv'), index=False)
table_viii.to_csv(os.path.join(project_path, 'results', 'table_viii_finetuned_ablation_k5.csv'), index=False)
table_ix.to_csv(os.path.join(project_path, 'results', 'table_ix_cosine_and_transfer.csv'), index=False)

print("=== Table VII (base model, k=5) ===")
print(table_vii.to_string(index=False))
print("\n=== Table VIII (fine-tuned model, k=5) ===")
print(table_viii.to_string(index=False))
print("\n=== Table IX (cosine similarity + ablation transfer) ===")
print(table_ix.to_string(index=False))

# Dose-response figure: target-class drop vs k, one subplot per model.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, model_name in zip(axes, ['base', 'finetuned']):
    for class_name in EMOTION_CLASSES:
        drops = [ablation_sweeps[model_name][class_name][k]['target_drop'] for k in K_VALUES]
        ax.plot(K_VALUES, drops, marker='o', label=class_name)
    ax.set_title(f'{model_name.title()} model')
    ax.set_xlabel('k (top-k neurons ablated)')
    ax.axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('Target-class accuracy drop (pp)')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
fig.suptitle('Dose-response: target-class accuracy drop vs. k (RML test split, N=144)')
fig.tight_layout()
fig_path = os.path.join(project_path, 'figures', 'dose_response_k_sweep.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\nSaved {fig_path}")
plt.show()

print("\nWeek 4 Days 16-18 artifacts complete: Table VII/VIII/IX CSVs in results/, "
      "dose-response figure in figures/. The written subsection is the manual step -- see Handoff.")

# Handoff

`results/` now contains: `day0_dominant_modality.json`, `week1_probe_auc.json`, `week2_fast_path_verification.json`, `week2_{base,finetuned}_ablation_sweep.json`, `week3_ablation_transfer.json`, `week3_transfer_retention.csv`, `table_{vii,viii,ix}_*.csv`. `figures/` has `dose_response_k_sweep.png`. `checkpoints/activations/` holds the cached tensors and their provenance stamps (resumable — Days 1–2 skip re-extraction when the stamp matches).

**Next steps — Week 4, Days 16–18 (manual write-up):**

1. Write the session journal following the 7-part schema in `journals/GUIDELINES.md`. Record the actual fidelity-gate accuracies, the Day 0 step 2 SHAP margins, whether the falsification pair passed, whether the fast path verified, the probe AUCs, and the Week 3 verdict.
2. Draft Section VI-D per protocol Days 16–18: the Week 2 ablation-selectivity result (causal validation, **scoped to RML only** — say so explicitly), then the Week 3 preservation-vs-reassignment result as the candidate explanation for RML's limited fine-tuning gain despite an unchanged dominant modality.
3. Update Limitation #1 to note this partial resolution, with the other five datasets as future work.
4. State the dominant-modality framing the way ADR 0005 records it: audio leads on **per-neuron** attribution, text leads on **aggregate** attribution mass because it has 16× the dimensions. Both are the same measurement. Do not write "two metrics disagree", and do not write "audio won a near-tie".

**Out of scope** per protocol: the other five datasets, the LIRIS-ACCEDE diffuse-activation diagnostic, any SAE dictionary (Days 19–20 optional stretch only), and cross-modality comparisons beyond the single dominant modality.